In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# Reading tables
ordered_products_df = spark.read.table("ecommerce_analytics.silver.orderd_products").alias("op")
promotions_df = spark.read.table("ecommerce_analytics.silver.promotions").alias("promo")
sales_order_df = spark.read.table("ecommerce_analytics.silver.sales_orders").alias("so")


def fact_table():
    df_so_op = ordered_products_df.join(sales_order_df,col("op.order_number") == col("so.order_number"),"left")\
        .select(col("op.order_number"),
                col("so.customer_id"),
                col("op.id").alias("product_id"),
                col("op.price").alias("unit_price"),
                col("op.qty").alias("quantity"),
                col("so.order_timestamps").alias("order_date_key")
        )

    df_final = df_so_op.join(promotions_df, col("op.order_number") == col("promo.order_number"), "left")\
        .select(
            col("op.order_number"),
            col("so.customer_id"),
            col("product_id"),
            col("unit_price"),
            col("quantity"),
            col("order_date_key"),
            col("promo.promo_qty").alias("promo_quantity"),
            col("promo.promo_disc").alias("promo_discount")
        )

    df_final = df_final.withColumn(
        "total_amount",
        col("quantity").cast("double") *
        col("unit_price").cast("double") *
        (1 - coalesce(col("promo_discount").cast("double"), lit(0.0)))
    )
    return df_final

fact_table().write.mode("overwrite").saveAsTable("ecommerce_analytics.gold.fact_table")

In [0]:
display(fact_table())